# Get started with ChakraTab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhatlabs/yhatlabs-cookbook/blob/main/getting-started/chakratab.ipynb)

ChakraTab predicts a column of a table from your labeled rows: a probability for every class, or a number with a range
around it. There is no model to train or tune. In this notebook you predict credit risk, compare with standard models,
see which columns matter, and predict house prices with intervals.

You need an API key: create one at https://yhatlabs.com/models/signup.

In [1]:
%pip install -q -U "yhatlabs[plot]" scikit-learn

In [2]:
import os
try:  # in Colab: add YHAT_API_KEY under Secrets (the key icon on the left)
    from google.colab import userdata
    os.environ.setdefault("YHAT_API_KEY", userdata.get("YHAT_API_KEY"))
except Exception:
    pass
if not os.environ.get("YHAT_API_KEY"):  # elsewhere: paste it here, or set it in your environment beforehand
    import getpass
    os.environ["YHAT_API_KEY"] = getpass.getpass("YHAT_API_KEY (from https://yhatlabs.com/models): ")

In [3]:
from yhatlabs import Client

yh = Client()  # reads YHAT_API_KEY

## 1. Predict a class

The German credit table: 1,000 loan applicants, each labeled as a good or bad credit risk.

In [4]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

credit = fetch_openml("credit-g", version=1, as_frame=True).frame
train, test = train_test_split(credit, test_size=0.2, random_state=0, stratify=credit["class"])
train.head(3)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
355,0<=X<200,24,existing paid,new car,1246,<100,<1,4,male single,none,...,real estate,23,stores,own,1,unskilled resident,1,none,yes,bad
661,<0,12,existing paid,new car,900,no known savings,1<=X<4,4,male mar/wid,none,...,car,23,none,own,1,skilled,1,none,yes,bad
300,no checking,6,existing paid,new car,672,<100,unemployed,1,female div/dep/mar,none,...,real estate,54,none,own,1,unemp/unskilled non res,1,yes,yes,good


In [5]:
model = yh.fit(train, target="class")
res = model.predict(test.drop(columns="class"))
res.df.head()

,prediction,p_bad,p_good
40,good,0.107157,0.892843
383,good,0.284451,0.715549
986,bad,0.593758,0.406242
79,bad,0.631039,0.368961
775,bad,0.689121,0.310879


In [6]:
from sklearn.metrics import accuracy_score, roc_auc_score

print(f"accuracy {accuracy_score(test['class'], res.df['prediction']):.3f}, "
      f"ROC AUC {roc_auc_score(test['class'] == 'bad', res.df['p_bad']):.3f}")

accuracy 0.795, ROC AUC 0.851


## 2. Compare with standard models

The same split, scored for ChakraTab and for gradient boosting and other standard models trained on it.

In [7]:
cmp = yh.predict(train, "class", test.drop(columns="class"), compare=True)
cmp.comparison.score(test["class"], ChakraTab=cmp)

,model,accuracy,balanced_accuracy,log_loss,roc_auc
0,ChakraTab,0.795,0.705952,0.431779,0.850595
1,Linear model,0.795,0.725000,0.457192,0.832262
2,Random forest,0.785,0.689286,0.460881,0.842321
3,XGBoost,0.765,0.679762,0.459348,0.825476
4,Most common class,0.700,0.500000,NaN,NaN


## 3. See which columns matter

In [8]:
model.importance(test)

,feature,importance,stddev
0,checking_status,0.114310,0.010156
1,duration,0.063670,0.015867
2,credit_history,0.026980,0.003168
3,purpose,0.026481,0.008158
4,credit_amount,0.023927,0.007564
5,other_payment_plans,0.013714,0.007160
6,savings_status,0.012085,0.004006
7,employment,0.010711,0.007785
8,age,0.007944,0.004104
9,property_magnitude,0.006697,0.002776


## 4. Predict a number with a range

California house prices (median value per district, in $100,000s). `quantiles=[0.05, 0.5, 0.95]` asks for a 90% range around every prediction.

In [9]:
from sklearn.datasets import fetch_california_housing

houses = fetch_california_housing(as_frame=True).frame.sample(5000, random_state=0)
train, test = train_test_split(houses, test_size=0.2, random_state=0)
model = yh.fit(train, target="MedHouseVal", task="regress")
res = model.predict(test.drop(columns="MedHouseVal"), quantiles=[0.05, 0.5, 0.95])
res.df.head()

,prediction,q5,median,q95,mean
11479,2.747073,2.236952,2.701673,3.361500,2.740218
5036,1.461847,1.188082,1.461804,1.749802,1.465708
2772,0.599216,0.260794,0.519018,1.129176,0.588500
17989,2.003514,0.319828,1.266512,4.161470,1.613108
17534,1.970561,1.598768,1.988322,2.458616,2.006664


In [10]:
y = test["MedHouseVal"].to_numpy()
inside = ((y >= res.df["q5"].to_numpy()) & (y <= res.df["q95"].to_numpy())).mean()
error = abs(res.df["median"].to_numpy() - y).mean()
print(f"mean absolute error {error:.3f}; {inside:.0%} of actual prices fall inside the 90% range")

mean absolute error 0.239; 89% of actual prices fall inside the 90% range


*Data: German credit ([OpenML 31](https://www.openml.org/d/31), from the UCI repository, CC BY 4.0); California housing (scikit-learn, from the 1990 US census).*

## Next steps

- [Docs](https://yhatlabs.com/models/docs): every option of `fit` and `predict`, and the scikit-learn estimators
- Questions: founders@yhatlabs.com